# Does ALL director history help?

Compare **tuned lasso ratings** and **three-class logistic financial predictions**, with versus without all **nine** earlier-film director features. The current movie’s director count remains a movie characteristic, not history.

Both arms use identical movies, non-director inputs and outer chronological folds inside the original training period. Each arm independently retunes the same parameter grid on three **inner** chronological folds. With-history makes all nine fields available; no history feature is removed by overlap trimming during this experiment. Other model-specific exclusions are applied equally to both arms.

Financial selection uses the shared **≥70% mean inner-validation hit precision, then maximum hit F1** rule. Infeasible folds retain and flag the original reference. Outer validation precision need not meet the inner selection goal. No later test outcomes select or score this experiment. Retrospective histories and reused data still make the findings exploratory.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../Modeling_scripts').resolve()))
from IPython.display import display
from director_history_comparison import run_comparison, summarize, save_comparison
from project_policy import DIRECTOR_HISTORY_FEATURES
print('History block:', ', '.join(DIRECTOR_HISTORY_FEATURES))

History block: dir_prior_success_rate_smoothed, dir_prior_films_log1p, dir_years_active_log1p, dir_prior_known_films_log1p, dir_first_film, dir_no_track_record, dir_prior_rating_smoothed, dir_prior_rated_films_log1p, dir_no_rating_history


In [2]:
fold_results = run_comparison()
summary = summarize(fold_results)
for task, table in summary.items():
    print(task.upper(), "— mean OUTER validation scores")
    display(table.round(4))
comparison_report = save_comparison(fold_results)
print("Saved audit details:", comparison_report)

13,276 movies | 9,405 training / 3,871 later test | 40 available predictors


rating Without director history outer fold 1 completed


rating Without director history outer fold 2 completed


rating Without director history outer fold 3 completed


rating With all director history outer fold 1 completed


rating With all director history outer fold 2 completed


... (9 outputs left out)

,MAE,RMSE,R2
History,,,
Without director history,0.5190,0.6689,0.3106
With all director history,0.5182,0.6631,0.3226


SUCCESS — mean OUTER validation scores


,Macro F1,Moderate recall,Hit precision,Hit recall,Hit F1,Precision goal met (folds)
History,,,,,,
Without director history,0.3743,0.3483,0.6492,0.1747,0.2585,1
With all director history,0.3874,0.4155,0.6694,0.1960,0.2834,1


Saved audit details: /Users/waltroou/DS-6021/DS_Final_Project/DS6021_Group2_FinalProject/Data_visualization/results/director_history_comparison.json


In [3]:
for task, metric in [('rating','MAE'), ('success','Macro F1'), ('success','Hit F1')]:
    paired = fold_results.loc[fold_results.Task.eq(task)].pivot(index='Fold',columns='History',values=metric)
    paired['Gain from director history'] = (
        paired['Without director history'] - paired['With all director history'] if metric=='MAE'
        else paired['With all director history'] - paired['Without director history'])
    print(task, metric, '— positive gain favors history')
    display(paired.round(5))
    print('Mean gain:', round(paired['Gain from director history'].mean(),6),
          '| across-fold SD:', round(paired['Gain from director history'].std(),6))

rating MAE — positive gain favors history


History,With all director history,Without director history,Gain from director history
Fold,,,
1,0.52169,0.52393,0.00224
2,0.51568,0.51752,0.00184
3,0.51717,0.51540,-0.00177


Mean gain: 0.000774 | across-fold SD: 0.00221
success Macro F1 — positive gain favors history


History,With all director history,Without director history,Gain from director history
Fold,,,
1,0.36363,0.35523,0.00840
2,0.43134,0.41317,0.01817
3,0.36725,0.35452,0.01273


Mean gain: 0.0131 | across-fold SD: 0.004898
success Hit F1 — positive gain favors history


History,With all director history,Without director history,Gain from director history
Fold,,,
1,0.29672,0.29379,0.00293
2,0.40000,0.34610,0.05390
3,0.15336,0.13565,0.01772


Mean gain: 0.024851 | across-fold SD: 0.026222


Interpret the paired differences and fold consistency, not just a tiny average change. This is a predictive ablation, not a causal effect or coefficient significance test. Do not claim an infeasible inner precision goal was achieved, and do not promote a model using the already-inspected later test period.

In the current run, each financial arm met the inner precision requirement in 1/3 outer training windows. The other two use the explicitly flagged original reference. Financial macro/hit F1 improved in all three outer folds; rating MAE improved in two and worsened in one.